# Caratterizzazione dei picchi e attivita

Fit Poisson simultanei sui canali originali, senza rebinning. Entrambe le definizioni di `FILES` sono conservate; la seconda resta disattivata come nell’originale. I CSV devono contenere conteggi grezzi interi nella prima colonna.

Le efficienze e le probabilita di emissione presenti sono riprese dal notebook: controllarle rispetto alla geometria del campione. I nomi candidati non costituiscono identificazioni validate. Il codice fitta tutte le righe elencate, senza dichiararle automaticamente rivelate. Le componenti non risolte possono essere fortemente correlate.

Le attivita sono medie temporali, senza correzione per decadimento, senza sottrazione di un blank e assumendo la stessa efficienza tra acquisizioni. Gli errori sono locali di Fisher, non limiti superiori o intervalli esatti per segnali deboli. Con incertezze relative di efficienza/BR e massa impostate a zero, l’errore totale include soltanto la statistica. Nessuna equivalenza con attivita dei progenitori viene assunta.

Output: panoramica, plot di ogni fit con componenti e residui, CSV delle aree e attivita, covarianze NPZ. Il doppietto di 234Th ha un risultato combinato dedicato.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import ndtr, xlogy

# CONFIGURAZIONE
FILES = {
    "50d": {
        "path": "csv-files/zro2_white&green_ 50_days.csv",
        "time": 4279386,
        "calibration": {
            "a": 0.357304,
            "b": -0.295444,
            "c": 1.352297e-9,
        },
    },
    
    "90d": {
        "path": "csv-files/90_days.csv",
        "time": 7550023,
        "calibration": {
            "a": 4.179423e-01,
            "b": 1.2696e-01,
            "c": 1.514025e-08,
        },
    },
    
}

# Seconda definizione alternativa: togliere le triple virgolette per usare solo 50d.
'''FILES = {
    "50d": {
        "path": "csv-files/zro2_white&green_ 50_days.csv",
        "time": 4279386,
        "calibration": {
            "a": 0.357304,
            "b": -0.295444,
            "c": 1.352297e-9,
        },
    }
    
}'''

OUTPUT_DIR = Path("background_results")
MASS_KG = 0.5513
MASS_U_KG = 0.0  # Inserire l'incertezza standard della massa se disponibile.
ENERGY_RANGE = (40., 3000.)
SHOW_PLOTS = True
# Fattore moltiplicativo della sigma originale: adattabile per acquisizione.
for cfg in FILES.values():
    cfg.setdefault("resolution_scale", 1.0)


def sigma_fun(E):
    """Sigma [keV]; formula del notebook, riscritta senza divisioni per E."""
    return np.sqrt(36.8436 * np.asarray(E) + 17271.4) / (2.355 * 100.)


# E, identificazione, efficienza, probabilita gamma per decadimento.
# Le efficienze e BR valorizzate provengono dal notebook, non da una nuova
# calibrazione. None = dato mancante, attivita NON calcolata.
# Le attribuzioni ambigue restano esplicitamente da verificare.
def line(E, isotope, eff=None, br=None, note="", eff_rel_u=0., br_rel_u=0.):
    return dict(E=float(E), isotope=isotope, eff=eff, br=br, note=note,
                eff_rel_u=eff_rel_u, br_rel_u=br_rel_u)


LINES = [
    line(91.70, "234Th", .005346, .0281),
    line(92.50, "234Th", .005346, .0277),
    line(185.72, "235U", note="Interferenza con 226Ra a 186.211 keV"),
    line(186.211, "226Ra", .032, .0359,
         "Attribuzione corretta rispetto a 226Rn; doppietto poco risolto"),
    line(205.31, "235U"),
    line(238.63, "212Pb"),
    line(295.224, "214Pb"),
    line(300.087, "212Pb", .0348, .0328),
    line(302.75, "da verificare"),
    line(330.10, "da verificare", note="Il nome 211Pa del notebook non e validato"),
    line(338.32, "228Ac"),
    line(351.059, "211Bi"),
    line(351.932, "214Pb", .018020, .334,
         "Efficienza copiata dal notebook: verificare, identica a quella di 40K"),
    line(401.81, "219Rn", .0318, .0637,
         "Identificazione da confermare; nel notebook chiamato 211Rn"),
    line(404.85, "211Pb", .0318, .0378),
    line(583.19, "208Tl"),
    line(609.312, "214Bi"),
    line(661.655, "137Cs", .026, .8501),
    line(724.20, "95Zr"),
    line(727.33, "212Bi", .0249, .0658),
    line(755.28, "228Ac", note="Energia candidata: verificare libreria gamma"),
    line(756.70, "95Zr", .0249, .544,
         "Fit simultaneo con 228Ac; verificare efficienza e interferenze"),
    line(763.13, "228Ac", note="Energia candidata: verificare libreria gamma"),
    line(765.80, "95Nb"),
    line(768.36, "214Bi"),
    line(778.20, "segnale candidato", note="Riga di interesse del notebook"),
    line(785.95, "214Pb / 212Bi", note="Identificazione ambigua: area combinata"),
    line(794.95, "228Ac", note="Energia candidata: verificare libreria gamma"),
    line(898.04, "88Y"),
    line(911.20, "228Ac"),
    line(968.97, "228Ac"),
    line(1001.31, "234mPa", .0215, .0084),
    line(1112.08, "152Eu", note="BR .0037 del notebook non trasferita: da verificare"),
    line(1120.287, "214Bi", .0204, .1510),
    line(1460.82, "40K", .018020, .107),
    line(1764.49, "214Bi"),
    line(1836.06, "88Y"),
    line(2614.50, "208Tl", .0123, .9916,
         "Attivita di 208Tl: nessuna conversione automatica alla catena di 232Th"),
]

# ROI non sovrapposte: i picchi vicini sono fittati contemporaneamente.
REGIONS = [
    (86, 98), (180, 192), (200, 211), (232, 245), (287, 310),
    (322, 344), (345, 359), (393, 413), (575, 591), (601, 618),
    (653, 670), (716, 735), (747, 802), (888, 906), (906, 920),
    (960, 978), (991, 1012), (1103, 1130), (1450, 1473),
    (1753, 1777), (1824, 1849), (2598, 2631),
]


def load_spectra(files):
    """Un conteggio per canale CSV; nessun rebinning nei fit Poisson.

    I bordi energetici sono ottenuti calibrando ch +/- 0.5. I dataset
    restano separati: nessun conteggio viene frazionato o duplicato.
    """
    spectra = {}
    for name, cfg in files.items():
        counts = pd.read_csv(cfg["path"], header=None).iloc[:, 0].to_numpy(float)
        if (not np.all(np.isfinite(counts)) or np.any(counts < 0)
                or not np.allclose(counts, np.rint(counts), atol=1e-7, rtol=0)):
            raise ValueError(f"{name}: richiesti conteggi grezzi interi non negativi.")
        if not np.isfinite(cfg['time']) or cfg['time'] <= 0:
            raise ValueError(f"{name}: live time non valido")
        c = cfg['calibration']
        ch = np.arange(counts.size + 1, dtype=float) - .5
        edges = c['b'] + c['a'] * ch + c['c'] * ch**2
        if np.any(np.diff(edges) <= 0):
            raise ValueError(f"{name}: calibrazione non monotona")
        spectra[name] = dict(counts=counts, edges=edges, time=float(cfg['time']),
                             scale=float(cfg['resolution_scale']))
    return spectra


def deviance_residual(y, model):
    model = np.maximum(model, 1e-14)
    d = 2 * (xlogy(y, y / model) - y + model)
    return np.sign(y-model) * np.sqrt(np.maximum(d, 0.))


def fit_region(spectra, roi, lines):
    """Aree indipendenti per run; shift e risoluzione indipendenti per run.

    Ogni componente e integrata esattamente nei bin. Il fondo lineare e
    parametrizzato mediante due densita positive agli estremi della ROI.
    La sigma segue sigma_fun(E) con fattore libero [0.65, 1.5].
    Nessuna ipotesi di attivita costante tra le acquisizioni.
    """
    selected = []
    p0, low, high, slices = [], [], [], {}
    k = len(lines)
    for name, s in spectra.items():
        e = s['edges']
        use = (e[:-1] >= roi[0]) & (e[1:] <= roi[1])
        left, right, y = e[:-1][use], e[1:][use], s['counts'][use]
        if y.size <= k + 4:
            raise ValueError(f"{name}: ROI {roi} con pochi canali")
        width = right - left
        density = np.maximum(np.median(y / width), .01)
        initial_areas = []
        for ln in lines:
            around = np.abs((left+right)/2-ln['E']) < 2*sigma_fun(ln['E'])*s['scale']
            initial_areas.append(max(float(np.sum(y[around]-density*width[around])), 1.))
        start = len(p0)
        p0.extend(initial_areas + [0., 1., density, density])
        low.extend([0.]*k + [-.8, .65, 1e-10, 1e-10])
        high.extend([np.inf]*k + [.8, 1.5, np.inf, np.inf])
        slices[name] = slice(start, len(p0))
        selected.append((name, left, right, y))

    def components(p, item):
        name, left, right, y = item
        q = p[slices[name]]
        area, shift, scale, b0, b1 = q[:k], *q[k:]
        mu = np.array([ln['E'] for ln in lines]) + shift
        sig = sigma_fun(np.array([ln['E'] for ln in lines])) * scale * spectra[name]['scale']
        peaks = area[:, None] * (ndtr((right[None, :]-mu[:, None])/sig[:, None])
                                 - ndtr((left[None, :]-mu[:, None])/sig[:, None]))
        slope = (b1-b0)/(roi[1]-roi[0])
        bg = b0*(right-left) + .5*slope*((right-roi[0])**2-(left-roi[0])**2)
        return peaks, bg

    def residual(p):
        out = []
        for item in selected:
            peaks, bg = components(p, item)
            out.append(deviance_residual(item[3], peaks.sum(axis=0)+bg))
        return np.concatenate(out)

    best = None
    # Due partenze sulla calibrazione riducono la sensibilita al punto iniziale.
    for delta in (0., .25, -.25):
        initial = np.array(p0)
        for sl in slices.values():
            initial[sl.start+k] = delta
        fit = least_squares(residual, initial, bounds=(low, high),
                            x_scale='jac', max_nfev=4000, ftol=1e-9,
                            xtol=1e-9, gtol=1e-7)
        if best is None or np.sum(fit.fun**2) < np.sum(best.fun**2):
            best = fit
    # Informazione di Fisher attesa: appropriata per conteggi Poisson.
    # J_model = derivata del modello; cov non riscalata per chi2 ridotto.
    def expectation(p):
        return np.concatenate([components(p, it)[0].sum(axis=0)+components(p, it)[1]
                               for it in selected])
    expected = expectation(best.x)
    jac = np.empty((expected.size, best.x.size))
    for j in range(best.x.size):
        h = 1e-4 * max(1., abs(best.x[j]))
        plus, minus = best.x.copy(), best.x.copy()
        plus[j] += h
        minus[j] = max(low[j], minus[j]-h)
        jac[:, j] = (expectation(plus)-expectation(minus))/(plus[j]-minus[j])
    weighted = jac / np.sqrt(np.maximum(expected[:, None], 1e-12))
    # Colonne scalate per identificare le degenerazioni senza confondere unita.
    colnorm = np.maximum(np.linalg.norm(weighted, axis=0), 1e-15)
    standardized = weighted / colnorm
    info = standardized.T @ standardized
    condition = np.linalg.cond(info)
    covariance = np.linalg.pinv(info, rcond=1e-12) / np.outer(colnorm, colnorm)
    unreliable = (not best.success) or condition > 1e10
    return dict(roi=roi, lines=lines, fit=best, cov=covariance,
                condition=condition, unreliable=unreliable, slices=slices,
                selected=selected, components=components,
                deviance=float(np.sum(best.fun**2)),
                ndof=int(best.fun.size-best.x.size))


def plot_fit(result, output_dir, show=True):
    n = len(result['selected'])
    fig, axes = plt.subplots(2, n, figsize=(7*n, 6), squeeze=False,
                             gridspec_kw={'height_ratios': [3, 1]}, sharex='col')
    for col, item in enumerate(result['selected']):
        name, left, right, y = item
        peaks, bg = result['components'](result['fit'].x, item)
        model = peaks.sum(axis=0)+bg
        edges = np.r_[left, right[-1]]
        ax, pull = axes[:, col]
        ax.stairs(y, edges, color='navy', label='Dati')
        ax.errorbar((left+right)/2, y, yerr=np.sqrt(np.maximum(y, 1)),
                    fmt='none', color='navy', alpha=.35)
        ax.stairs(model, edges, color='deepskyblue', lw=2, label='Fit Poisson')
        ax.stairs(bg, edges, color='gray', ls=':', label='Fondo')
        for ln, pk in zip(result['lines'], peaks):
            ax.stairs(pk+bg, edges, ls='--', lw=1,
                      label=f"{ln['isotope']} {ln['E']:.2f}")
        ax.set(ylabel='Conteggi / canale originale', title=name)
        ax.legend(fontsize=8, ncol=2)
        pull.axhline(0, color='black', lw=1)
        pull.axhspan(-2, 2, color='gray', alpha=.15)
        pull.plot((left+right)/2, deviance_residual(y, model), '.', ms=4)
        pull.set(xlabel='Energia [keV]', ylabel='Residuo\ndevianza')
        for a in (ax, pull):
            a.grid(alpha=.2)
    title = f"ROI {result['roi']} keV | D/ndof = {result['deviance']:.1f}/{result['ndof']}"
    if result['unreliable']:
        title += ' | FIT / COVARIANZA DA CONTROLLARE'
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(output_dir / f"fit_{result['roi'][0]:04.0f}_{result['roi'][1]:04.0f}.png", dpi=180)
    if show:
        plt.show()
    plt.close(fig)


def extract_rows(result, spectra):
    rows = []
    k = len(result['lines'])
    for j, ln in enumerate(result['lines']):
        indices = [sl.start+j for sl in result['slices'].values()]
        N = float(result['fit'].x[indices].sum())
        var = float(result['cov'][np.ix_(indices, indices)].sum())
        u = np.sqrt(max(var, 0.)) if not result['unreliable'] else np.nan
        t = sum(s['time'] for s in spectra.values())
        near_boundary = any(result['fit'].active_mask[i] != 0 for i in indices)
        shape_boundary = any(np.any(result['fit'].active_mask[sl.start+k:sl.start+k+2])
                             for sl in result['slices'].values())
        status = ('fit/covarianza da controllare' if result['unreliable'] else
                  'area al limite: errore simmetrico non affidabile' if near_boundary else
                  'forma al limite: controllare calibrazione/risoluzione' if shape_boundary else
                  'debole: non e una rivelazione' if N < 3*u else 'area positiva; verificare identificazione')
        row = dict(isotope=ln['isotope'], energy_keV=ln['E'], N=N, u_N=u,
                   rate_cps=N/t, u_rate_cps=u/t, live_time_s=t,
                   efficiency=ln['eff'], emission_probability=ln['br'],
                   activity_mBq_kg=np.nan, u_activity_stat_mBq_kg=np.nan,
                   u_activity_total_mBq_kg=np.nan, status=status, note=ln['note'],
                   deviance=result['deviance'], ndof=result['ndof'])
        for name, sl in result['slices'].items():
            idx = sl.start+j
            row[f'N_{name}'] = result['fit'].x[idx]
            row[f'u_N_{name}'] = (np.sqrt(max(result['cov'][idx, idx], 0.))
                                   if not result['unreliable'] else np.nan)
        if ln['eff'] is not None and ln['br'] is not None:
            if not (0 < ln['eff'] <= 1 and 0 < ln['br'] <= 1):
                raise ValueError(f"Efficienza/BR non valida: {ln}")
            den = t*MASS_KG*ln['eff']*ln['br']
            a, ua = 1000*N/den, 1000*u/den
            row['activity_mBq_kg'] = a
            row['u_activity_stat_mBq_kg'] = ua
            row['u_activity_total_mBq_kg'] = np.sqrt(ua**2 + a**2*(
                (MASS_U_KG/MASS_KG)**2 + ln['eff_rel_u']**2 + ln['br_rel_u']**2))
        rows.append(row)
    return rows


def main():
    if MASS_KG <= 0:
        raise ValueError('Massa non valida')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    spectra = load_spectra(FILES)
    fig, ax = plt.subplots(figsize=(15, 4))
    for name, s in spectra.items():
        centers = (s['edges'][:-1]+s['edges'][1:])/2
        ax.plot(centers, s['counts']/s['time']/np.diff(s['edges']),
                lw=.6, label=name)
    ax.set(xlim=ENERGY_RANGE, yscale='log', xlabel='Energia [keV]',
           ylabel='Conteggi / (s keV)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'spectrum.png', dpi=180)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    rows, fits, failures = [], [], []
    for roi in REGIONS:
        lines = [ln for ln in LINES if roi[0] < ln['E'] < roi[1]]
        print(f"Fit {roi}: {len(lines)} componenti", flush=True)
        try:
            result = fit_region(spectra, roi, lines)
        except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
            failures.append(dict(roi=roi, error=str(exc)))
            warnings.warn(f'ROI {roi}: {exc}')
            continue
        fits.append(result)
        # Conserva parametri e covarianza per somme/doppietti e analisi successive.
        np.savez(OUTPUT_DIR/f"covariance_{roi[0]}_{roi[1]}.npz",
                 parameters=result['fit'].x, covariance=result['cov'],
                 energies=np.array([ln['E'] for ln in lines]),
                 datasets=np.array(list(result['slices'])))
        rows.extend(extract_rows(result, spectra))
        plot_fit(result, OUTPUT_DIR, SHOW_PLOTS)
    table = pd.DataFrame(rows)
    if not table.empty:
        table = table.sort_values('energy_keV').reset_index(drop=True)
        table.to_csv(OUTPUT_DIR/'all_peak_results.csv', index=False)
        activities = table[table['activity_mBq_kg'].notna()]
        activities.to_csv(OUTPUT_DIR/'activities.csv', index=False)
        print(activities[['isotope','energy_keV','N','u_N','activity_mBq_kg',
                          'u_activity_stat_mBq_kg','status']].to_string(index=False))
        # 234Th: somma delle aree con covarianza completa, evita media di
        # attivita fortemente anticorrelate del doppietto non risolto.
        for res in fits:
            if [ln['E'] for ln in res['lines']] == [91.7, 92.5]:
                ids = [sl.start+j for sl in res['slices'].values() for j in range(2)]
                area = res['fit'].x[ids].sum()
                uarea = np.sqrt(max(res['cov'][np.ix_(ids, ids)].sum(), 0.))
                if res['unreliable']:
                    uarea = np.nan
                norm = MASS_KG*sum(s['time'] for s in spectra.values()) * sum(
                    ln['eff']*ln['br'] for ln in res['lines'])
                combined = pd.DataFrame([dict(isotope='234Th', N=area, u_N=uarea,
                    activity_mBq_kg=1000*area/norm,
                    u_activity_stat_mBq_kg=1000*uarea/norm,
                    note='Media temporale; covarianza inclusa; errori locali di Fisher')])
                combined.to_csv(OUTPUT_DIR/'234Th_doublet_activity.csv', index=False)
                print('\nDoppietto 234Th:\n', combined.to_string(index=False))
    (OUTPUT_DIR/'failed_regions.json').write_text(json.dumps(failures, indent=2))
    print('\nAttivita medie sui live time. Efficienza comune tra run assunta.')
    print('Errori locali di Fisher: per picchi deboli/limiti servono intervalli di profilo.')
    print('Nessuna sottrazione di un blank: le attivita possono includere il fondo ambientale.')
    return table, fits


if __name__ == '__main__':
    results, fit_results = main()
